# Waxing Tests for `new_reflection.py`

This notebook tests only waxing-block reconstruction. Waning blocks are intentionally excluded for now.

For these tests, reflection rows are derived directly from the inversion matrix so starred different-block evidence is preserved.

In [21]:
import importlib

import TITO_Explore.new_reflection as nr
from TITO_Explore.types import TranslationInvariantTotalOrder
from TITO_Explore.inversion_set import print_inversion_matrix, print_inversion_set, tito_to_inversion_set

importlib.reload(nr)

from TITO_Explore.new_reflection import (
    block_waxing_waning,
    determine_block_order,
    determine_blocks,
    determine_pair_cases,
    reconstruct_block_values,
    reflection_rows_to_blocks_and_order,
)


In [22]:
def reflection_rows_from_inversion_matrix(inversion_matrix):
    rows = []
    for start, row in enumerate(inversion_matrix):
        for cell in row:
            if cell is None:
                continue
            offsets, is_starred = cell
            for offset in offsets:
                rows.append((start, start + offset, 1 if is_starred else 0))
    return rows


def print_reflection_rows(rows):
    print(f"{'start':<8} {'target':<8} {'indicator':<10}")
    print("-" * 30)
    for start, target, indicator in rows:
        print(f"{start:<8} {target:<8} {indicator:<10}")


def run_waxing_case(name, n, vectors, expected_blocks):
    tito = TranslationInvariantTotalOrder(
        n=n,
        num_blocks=len(vectors),
        vectors=vectors,
        waxing_waning=[0] * len(vectors),
    )

    print(f"=== {name} ===")
    print("Input TITO:", tito)
    print("Expected reconstructed blocks:", expected_blocks)

    inversion_matrix = tito_to_inversion_set(tito)
    rows = reflection_rows_from_inversion_matrix(inversion_matrix)

    print("\nInversion matrix:")
    print_inversion_matrix(inversion_matrix)

    print("\nInversion set:")
    print_inversion_set(inversion_matrix)

    print("\nReflection rows:")
    print_reflection_rows(rows)

    pair_cases, diagonal_stars = determine_pair_cases(rows, tito.n)
    residue_blocks, residue_to_block = determine_blocks(pair_cases, tito.n)
    ordered_residue_blocks, block_report = determine_block_order(
        pair_cases,
        residue_blocks,
        residue_to_block,
    )
    waxing_waning = block_waxing_waning(ordered_residue_blocks, pair_cases, diagonal_stars)
    reconstructed_blocks, value_report = reconstruct_block_values(
        ordered_residue_blocks,
        waxing_waning,
        pair_cases,
        tito.n,
    )
    wrapper_result = reflection_rows_to_blocks_and_order(rows, tito.n)

    print("\nPair cases:")
    for pair, case in sorted(pair_cases.items()):
        print(pair, case)

    print("\nResidue blocks:", residue_blocks)
    print("Residue to block:", residue_to_block)
    print("Ordered residue blocks:", ordered_residue_blocks)
    print("Waxing/Waning:", waxing_waning)
    print("Block report:", block_report)
    print("Value report:", value_report)
    print("Reconstructed blocks:", reconstructed_blocks)
    print("Wrapper result:", wrapper_result)

    assert reconstructed_blocks == expected_blocks
    assert wrapper_result[0] == expected_blocks
    assert wrapper_result[1] == [0] * len(expected_blocks)

    return wrapper_result


## Single Waxing Block Cases

In [23]:
result_12 = run_waxing_case("[1,2]", 2, [[1, 2]], [[0, 1]])
result_16 = run_waxing_case("[1,6]", 2, [[1, 6]], [[0, -3]])
result_05 = run_waxing_case("[0,5]", 2, [[0, 5]], [[0, 5]])
result_021 = run_waxing_case("[0,2,1]", 3, [[0, 2, 1]], [[0, 2, 1]])
result_051 = run_waxing_case("[0,5,1]", 3, [[0, 5, 1]], [[0, 5, 1]])


=== [1,2] ===
Input TITO: TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[1, 2]], waxing_waning=[0])
Expected reconstructed blocks: [[0, 1]]
process result (0,1)

Inversion matrix:
         .          .
         .          .

Inversion set:
{  }

Reflection rows:
start    target   indicator 
------------------------------

Pair cases:
(0, 1) PairCase(pair=(0, 1), case='same_block_waxing', evidence=(), before=None)

Residue blocks: [[0, 1]]
Residue to block: {0: 0, 1: 0}
Ordered residue blocks: [[0, 1]]
Waxing/Waning: [0]
Block report: None
Value report: None
Reconstructed blocks: [[0, 1]]
Wrapper result: ([[0, 1]], [0], None)
=== [1,6] ===
Input TITO: TranslationInvariantTotalOrder(n=2, num_blocks=1, vectors=[[1, 6]], waxing_waning=[0])
Expected reconstructed blocks: [[0, -3]]
process result (0,-3)

Inversion matrix:
         .          .
    [1, 3]          .

Inversion set:
{ (1,2), (1,4) }

Reflection rows:
start    target   indicator 
------------------------------
1   

## Multiple Waxing Block Cases

In [24]:
result_multi_1 = run_waxing_case(
    "n=3, [0,2][1]",
    3,
    [[0, 2], [1]],
    [[0, 2], [1]],
)

result_multi_2 = run_waxing_case(
    "n=3, [1][0,2]",
    3,
    [[1], [0, 2]],
    [[1], [0, 2]],
)

result_multi_3 = run_waxing_case(
    "n=4, [0,3][1,2]",
    4,
    [[0, 3], [1, 2]],
    [[0, 3], [1, 2]],
)

result_multi_4 = run_waxing_case(
    "n=4, [1,6][0,3]",
    4,
    [[1, 6], [0, 3]],
    [[1, 6], [0, 3]],
)


=== n=3, [0,2][1] ===
Input TITO: TranslationInvariantTotalOrder(n=3, num_blocks=2, vectors=[[0, 2], [1]], waxing_waning=[0, 0])
Expected reconstructed blocks: [[0, 2], [1]]
process result (0,2)

Inversion matrix:
         .          .          .
      [2]*          .       [1]*
         .          .          .

Inversion set:
{ (1,3)*, (1,2)* }

Reflection rows:
start    target   indicator 
------------------------------
1        3        1         
1        2        1         

Pair cases:
(0, 1) PairCase(pair=(0, 1), case='different_blocks', evidence=((1, 3, 1, 0, True),), before=(0, 1))
(0, 2) PairCase(pair=(0, 2), case='same_block_waxing', evidence=(), before=None)
(1, 2) PairCase(pair=(1, 2), case='different_blocks', evidence=((1, 2, 1, 2, True),), before=(2, 1))

Residue blocks: [[0, 2], [1]]
Residue to block: {0: 0, 2: 0, 1: 1}
Ordered residue blocks: [[0, 2], [1]]
Waxing/Waning: [0, 0]
Block report: None
Value report: None
Reconstructed blocks: [[0, 2], [1]]
Wrapper result: ([